# Patrones de desarrollo de los países — PCA y Clustering (WDI)

**TP Final — Análisis Multivariado y Descubrimiento de Patrones (Universidad Austral).**

Este notebook **consume** los resultados ya generados por el pipeline (`src/run_all.py`):
lee de `data/processed/` y `reports/figures/` y arma la narrativa. **No recalcula** todo.

Corte moderno **2023** (año macro estable, no post-COVID) y comparación con **2005**.

> Ejecutar antes: `python src/run_all.py`

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
FIG = ROOT / 'reports' / 'figures'
YM = 2023  # año moderno
pd.set_option('display.max_columns', 30)
print('Resultados desde:', PROC)

## 1. Datos y cobertura

WDI del Banco Mundial, ~191 países, corte 2023 (+ 2005). **Año moderno = 2023** elegido por estabilidad macro (2021=rebote post-COVID, 2022=shock inflacionario) con cobertura adecuada. Variables descartadas: Gini, gasto en educación, INB-PPA.

In [ ]:
cov = pd.read_csv(PROC / 'cobertura_por_anio.csv')
cov[['label', '2000', '2005', '2021', '2023']]

## 2. EDA: transformaciones y correlación

Transformación decidida variable por variable según la asimetría observada.

In [ ]:
display(pd.read_csv(PROC / 'skewness_antes_despues.csv'))
display(Image(filename=str(FIG / f'eda_correlacion_{YM}.png')))

## 3. PCA

Retención por **análisis paralelo de Horn** (3 componentes). PC1 = gradiente de desarrollo (~41%). Nota: el PBI per cápita es **real** (constant 2015 US$).

In [ ]:
info = json.load(open(PROC / f'pca_info_{YM}.json', encoding='utf-8'))
print('Retención -> Kaiser:', info['kaiser'], '| 80% var:', info['n_80pct'], '| Horn:', info['parallel_analysis'])
print('Componentes para clusterizar:', info['n_clust'], '| para visualizar:', info['n_vis'])
display(Image(filename=str(FIG / f'pca_scree_{YM}.png')))

In [ ]:
load = pd.read_csv(PROC / f'pca_loadings_{YM}.csv', index_col=0)
display(load.iloc[:, :3].round(2))
display(Image(filename=str(FIG / f'pca_biplot_ingreso_{YM}.png')))

## 4. Clustering

Principal: k-means sobre las **14 variables**. Comparación con PCA: **ARI 1,00** (la partición es la misma → el *tandem analysis* no sesga aquí). k=2 por consenso de métricas (incl. gap) + estabilidad.

In [ ]:
res = json.load(open(PROC / f'clustering_resultados_{YM}.json', encoding='utf-8'))
print('Comparaciones k=2:', json.dumps(res['comparaciones']['2'], indent=2, ensure_ascii=False))
print('Estabilidad (Jaccard):', res['estabilidad'])
print('Alternativas (GMM, HDBSCAN):', res['alternativas'])
display(Image(filename=str(FIG / f'clust_metricas_k_{YM}.png')))

In [ ]:
print('Perfil de clusters (medianas, variables crudas) — k=2:')
display(pd.read_csv(PROC / f'perfil_medianas_k2_{YM}.csv', index_col=0).round(1))
print('Cruce cluster x nivel de ingreso:')
display(pd.read_csv(PROC / f'cruce_ingreso_k2_{YM}.csv', index_col=0))
display(Image(filename=str(FIG / f'clust_pca_k2_{YM}.png')))

## 5. Comparación temporal 2005 vs 2023

Espacio común (un solo PCA sobre datos apilados). Casi todos los países avanzaron; 40 'graduaron' de cluster, ninguno regresó. La estructura (PC1) es muy estable (congruencia 0,96).

In [ ]:
comp = json.load(open(PROC / 'compare_resultados.json', encoding='utf-8'))
print(json.dumps(comp, indent=2, ensure_ascii=False))
display(Image(filename=str(FIG / 'compare_trayectorias.png')))
display(Image(filename=str(FIG / 'compare_transiciones.png')))

### 5.a ¿PBI nominal distorsiona la lectura del avance?

**No.** (1) El PBI per cápita es **real** (constant 2015 US$). (2) Descomponiendo el avance medio en PC1, **Internet (47%)** y **esperanza de vida (22%)** lo dominan; el PBI real aporta solo **9%**. El avance es desarrollo genuino (conectividad, salud), no un artefacto de precios.

In [ ]:
display(pd.read_csv(PROC / 'descomposicion_dPC1.csv'))
display(Image(filename=str(FIG / 'compare_descomposicion_dPC1.png')))

In [ ]:
traj = pd.read_csv(PROC / 'trayectorias.csv')
print('Mayores avances en PC1 (gradiente de desarrollo):')
display(traj.sort_values('dPC1', ascending=False).head(8)[['country', 'cluster_e', 'cluster_m', 'dPC1']].round(2))
print('Mayores retrocesos:')
display(traj.sort_values('dPC1').head(5)[['country', 'cluster_e', 'cluster_m', 'dPC1']].round(2))

## 6. Sensibilidad

Robusto a imputación, nº de componentes y outliers. Único quiebre: **RobustScaler**, por amplificar el outlier de crecimiento de Macao (+75% en 2023) → justifica usar StandardScaler.

In [ ]:
display(pd.read_csv(PROC / 'sensibilidad.csv'))

## 7. Conclusiones

1. **PC1 (~41%)** resume el desarrollo como un **gradiente** (ingreso, salud, urbanización, conectividad vs peso agrario); PC2 = estructura productiva; PC3 = apertura/macro.
2. Clustering: **macro-separación robusta en 2 grupos** (desarrollado / en desarrollo), alineada con el ingreso del Banco Mundial, con excepciones informativas. El desarrollo es un **continuo**, no una taxonomía fina.
3. **2005→2023**: casi todos avanzaron (impulsado por **conectividad y salud**, no por el PBI —que además es real—); 40 graduaron, ninguno regresó; estructura muy estable (congruencia PC1 = 0,96).
4. Clusterizar sobre variables completas y **comparar** con PCA evita el sesgo del *tandem analysis* (aquí coinciden, ARI 1,00 — demostrado, no asumido).

Ver `reports/informe_borrador.md` y `reports/decisiones_metodologicas.md`.